# Intrinsics Check: Z4 vs CCD Height Map (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-20  
**Last Modified:** 2026-04-20  
**Status:** In Progress  
**Keywords:** AOS, Intrinsic Wavefront, Z4, Defocus, CCD Height Map, Full Array Mode  

## Description

Check whether the Z4 intrinsic wavefront map (as produced by the AOS pipeline and
stored in the FAM mktable HDF5 output) correctly accounts for the focal plane
CCD-to-CCD height variation.

Key functionality:
1. Read the FAM donuts + visits HDF5 produced by `intrinsics_mktable` (or the `mktable` step of `run-pipeline`)
2. Read the matching `_fits.parquet` file with the per-visit linear (k1, k2, k3) fit coefficients
3. Subtract the linear (tilt/tip + piston) portion of Z4 per donut using OCS field angles
4. Bin the linear-corrected Z4 in focal plane (fpx, fpy) and plot the median map
5. Build a custom Z4 intrinsic per donut: radial polynomial (from `plot_Z_FAM_August-archive`) plus a CCD height map correction (15 μm Z4 per mm of piston)
6. Plot the custom Z4 intrinsic, the difference vs the linear-corrected data, and direct comparisons against the pipeline `zk_intrinsic` already stored in the HDF5

**Output:** In-notebook plots only (median maps, difference map, hexbin and 2D comparisons).

**Based on:**
- `~/Astrophysics/Code/Claude/plot_Z_FAM_August-archive.ipynb` — radial Z4 intrinsic polynomial, height map reader, Z4↔height conversion
- `intrinsics_mktable.ipynb` / `code/intrinsics_lib.py` — HDF5 donut table format
- `code/dz_fitting.py` — focal plane Zernike basis used by the `_fits.parquet` fits (fp_radius = 1.75°, basis normalized to unit disk)

<a id='changelog'></a>
## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-20 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Input HDF5 from intrinsics_mktable (streaming format)
INPUT_HDF5 = 'output/aos_fam_danish_triplets_20260315_20260317.hdf5'

# Corresponding linear-fit parquet (auto-derived as {stem}_fits.parquet if None)
FIT_PARQUET = None

# CCD focal plane height map (on USDF, copied under notebooks/rubin-work/aos/output/)
HEIGHT_MAP_FITS = 'output/LSST_FP_cold_b_measurement_4col_bysurface.fits'

# Coordinate system used for the linear fit (must match the fit parquet)
COORD_SYS = 'OCS'

# Focal-plane radius used for normalizing the Zernike basis in the fit
# (matches default in code/dz_fitting.py: focal_plane_zernike_basis)
FP_RADIUS_DEG = 1.75

# i-band radial polynomial for Z4 intrinsic (meters vs field radius in degrees)
# Copied from plot_Z_FAM_August-archive.ipynb
Z4_POLY_PARAMS_IBAND = [
    -4.343587395401283e-08,
     1.3885890937287275e-09,
     1.1110986388995441e-07,
     1.3827292490327174e-08,
    -5.0371687781156236e-08,
     6.458311579945517e-09,
]

# Height-to-Z4 conversion: μm of Z4 per mm of local piston (height).
# Guillem's estimate (≈0.15 μm Z4 per 10 μm defocus = 15 μm/mm) — same as old notebook.
HEIGHT_TO_Z4_UM_PER_MM = 15.0

# Binning for the focal-plane (fpx, fpy) median maps
FP_NSTEPS = 65            # number of bin edges
FP_MAX_MM = 320.0         # half-range in mm

# Whether to restrict to donuts whose intra and extra centroids matched
REQUIRE_MATCHED = True

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits import axes_grid1

from astropy.io import fits
from astropy.table import QTable, Table
from scipy.stats import binned_statistic_2d
from sklearn.neighbors import KNeighborsRegressor
import tqdm

# Pull in the mktable library so we can reuse restack_array_columns and the HDF5 reader
sys.path.insert(0, 'code')
from intrinsics_lib import read_donuts_table  # noqa: E402

plt.rcParams['figure.dpi'] = 110

<a id='functions'></a>
## Helper Functions

In [ ]:
def add_colorbar(im, aspect=20, pad_fraction=0.5, **kwargs):
    """Attach a vertical colorbar that matches the host axes' height."""
    divider = axes_grid1.make_axes_locatable(im.axes)
    width = axes_grid1.axes_size.AxesY(im.axes, aspect=1.0 / aspect)
    pad = axes_grid1.axes_size.Fraction(pad_fraction, width)
    current_ax = plt.gca()
    cax = divider.append_axes('right', size=width, pad=pad)
    plt.sca(current_ax)
    return im.axes.figure.colorbar(im, cax=cax, **kwargs)


def z4_radial_intrinsic_um(r_deg, params=Z4_POLY_PARAMS_IBAND):
    """Radial model for the Z4 intrinsic wavefront (returns μm).

    Parameters
    ----------
    r_deg : ndarray
        Field radius in degrees.
    params : sequence of 6 floats
        Polynomial coefficients [c0, c1, c2, c3, c4, c5] in meters,
        as in plot_Z_FAM_August-archive.ipynb.
    """
    c0, c1, c2, c3, c4, c5 = params
    z4_m = c0 + c1 * r_deg + c2 * r_deg**2 + c3 * r_deg**3 + c4 * r_deg**4 + c5 * r_deg**5
    return z4_m * 1.0e6


def make_metrology_table(file, rsid=None):
    """Build a per-point focal-plane metrology table from the height map FITS file.

    Mirrors the reader in plot_Z_FAM_August-archive.ipynb: each per-sensor BinTableHDU
    is concatenated into one table with columns (fpx, fpy, z_mod, z_meas, det).
    Note the x/y swap from CCS to focal-plane convention: fpx = Y_CCS, fpy = X_CCS.
    """
    rows = []
    with fits.open(file) as hdulist:
        for hdu in tqdm.tqdm(hdulist, desc='metrology HDUs'):
            if not isinstance(hdu, fits.BinTableHDU):
                continue
            extname = hdu.header.get('EXTNAME', '')
            if rsid is not None and extname != rsid:
                continue
            if rsid is None and not re.fullmatch(r'R\d\dS\d\d', extname):
                continue
            det_label = re.sub(r'(R\d\d)(S\d\d)', r'\1_\2', extname)
            tab = Table(hdu.data)
            for x, y, z_mod, z_meas in zip(
                tab['X_CCS'], tab['Y_CCS'], tab['Z_CCS_MODEL'], tab['Z_CCS_MEASURED']
            ):
                fpx = y  # fpx = Y_CCS
                fpy = x  # fpy = X_CCS
                rows.append([fpx, fpy, z_mod, z_meas, det_label])
    return Table(rows=rows, names=['fpx', 'fpy', 'z_mod', 'z_meas', 'det'])


def get_height_interpolator(metrology_table, k=3, weight_type='distance', ztype='measured'):
    """KNN interpolator for focal-plane height given a metrology table."""
    x = np.column_stack((metrology_table['fpx'], metrology_table['fpy']))
    if ztype == 'measured':
        y = np.array(metrology_table['z_meas'])
    elif ztype == 'model':
        y = np.array(metrology_table['z_mod'])
    else:
        raise ValueError("ztype must be 'measured' or 'model'")
    knn = KNeighborsRegressor(n_neighbors=k, weights=weight_type)
    knn.fit(x, y)

    def interp_func(fpx, fpy):
        fpx = np.atleast_1d(fpx)
        fpy = np.atleast_1d(fpy)
        pts = np.column_stack((fpx, fpy))
        return knn.predict(pts)

    return interp_func


def linear_fit_values(thx_deg, thy_deg, k1, k2, k3, fp_radius=FP_RADIUS_DEG):
    """Evaluate the Z1-Z3 focal-plane Zernike linear model at each donut.

    Matches the basis in code/dz_fitting.focal_plane_zernike_basis:
        Z1 = 1            (piston)
        Z2 = 2 * x        (tilt), x = thx_deg / fp_radius
        Z3 = 2 * y        (tip),  y = thy_deg / fp_radius
    So the linear model value is:
        k1 * 1 + k2 * 2*x + k3 * 2*y  (μm)
    """
    x = thx_deg / fp_radius
    y = thy_deg / fp_radius
    return k1 + 2.0 * k2 * x + 2.0 * k3 * y


def bin_median_2d(xval, yval, zval, xbins, ybins):
    """Median of zval on an (xbins × ybins) 2D grid over (xval, yval)."""
    stat, x_edges, y_edges, _ = binned_statistic_2d(
        xval, yval, zval, statistic='median', bins=[xbins, ybins]
    )
    return stat, x_edges, y_edges


def plot_fp_map(stat, x_edges, y_edges, title, cmap='viridis',
                vmin=None, vmax=None, label=None):
    """Imshow a (fpx, fpy) 2D map with a matched colorbar."""
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(
        stat.T, origin='lower',
        extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
        cmap=cmap, interpolation='none', aspect='equal',
        vmin=vmin, vmax=vmax,
    )
    ax.set_xlabel('fpx [mm]')
    ax.set_ylabel('fpy [mm]')
    ax.set_title(title)
    cb = add_colorbar(im)
    if label is not None:
        cb.set_label(label)
    fig.tight_layout()
    return fig, ax

<a id='data'></a>
## Data Access

In [ ]:
# Resolve paths and load donuts + visits from HDF5
hdf5_path = Path(INPUT_HDF5)
if not hdf5_path.exists():
    raise FileNotFoundError(f'HDF5 not found: {hdf5_path}')

if FIT_PARQUET is None:
    fits_path = hdf5_path.parent / f'{hdf5_path.stem}_fits.parquet'
else:
    fits_path = Path(FIT_PARQUET)
if not fits_path.exists():
    raise FileNotFoundError(f'Fit parquet not found: {fits_path}')

height_path = Path(HEIGHT_MAP_FITS)
if not height_path.exists():
    raise FileNotFoundError(f'Height map not found: {height_path}')

print(f'HDF5:        {hdf5_path}')
print(f'Fit parquet: {fits_path}')
print(f'Height map:  {height_path}')

aosTable = read_donuts_table(str(hdf5_path))
visit_info = QTable.read(str(hdf5_path), path='visits')
print(f'Loaded {len(aosTable):,} donuts, {len(visit_info)} visits')

In [ ]:
# Load the per-visit linear-fit coefficients (k1, k2, k3 for Z4)
fits_table = pd.read_parquet(fits_path)
print(f'Fit rows: {len(fits_table)}, columns: {len(fits_table.columns)}')
print([c for c in fits_table.columns if c.startswith('z1toz3_z4_')])

required = ['day_obs', 'seq_num',
            'z1toz3_z4_c1', 'z1toz3_z4_c2', 'z1toz3_z4_c3']
missing = [c for c in required if c not in fits_table.columns]
if missing:
    raise KeyError(f'Missing required fit columns: {missing}')

if 'z1toz3_bad_fit' in fits_table.columns:
    n_bad = int(fits_table['z1toz3_bad_fit'].sum())
    print(f'  bad_fit visits: {n_bad}/{len(fits_table)}')

<a id='analysis'></a>
## Analysis

In [ ]:
# Build donut-level arrays we will re-use
zk_col = f'zk_{COORD_SYS}'
zk_intr_col = f'zk_intrinsic_{COORD_SYS}'  # pipeline's per-donut intrinsic (μm)
thx_col = f'thx_{COORD_SYS}'
thy_col = f'thy_{COORD_SYS}'

zk_arr = np.stack(aosTable[zk_col])          # (n_donuts, n_Z)
nZk = zk_arr.shape[1]
iZs = list(range(4, 4 + nZk)) if nZk != 19 else list(range(4, 23))
if 'nollIndices' in visit_info.colnames:
    iZs_vi = [int(n) for n in visit_info['nollIndices'][0]]
    if len(iZs_vi) == nZk:
        iZs = iZs_vi
iZidx = {iZ: i for i, iZ in enumerate(iZs)}
print(f'Noll indices ({nZk} terms): {iZs}')
z4_col_idx = iZidx[4]

Z4_data_um = zk_arr[:, z4_col_idx]
Z4_pipeline_intrinsic_um = np.stack(aosTable[zk_intr_col])[:, z4_col_idx]

# Field angles in OCS (linear fits use these) — already in radians
thx_deg = np.rad2deg(np.array(aosTable[thx_col]))
thy_deg = np.rad2deg(np.array(aosTable[thy_col]))

# fpx / fpy in mm — use the intra-focal centroid (either intra or extra would work;
# matched donuts have both within ~100 mas of each other in the OCS field).
fpx = np.array(aosTable['intra_fpx'])
fpy = np.array(aosTable['intra_fpy'])

# Optional selection
keep = np.ones(len(aosTable), dtype=bool)
if REQUIRE_MATCHED and 'matched_intra_extra' in aosTable.colnames:
    keep &= np.array(aosTable['matched_intra_extra'])
print(f'Selecting {keep.sum():,}/{len(keep):,} donuts')

In [ ]:
# Merge per-visit (k1, k2, k3) coefficients onto the donut table and
# compute the linear-fit Z4 prediction per donut.

fit_keys = fits_table[['day_obs', 'seq_num',
                       'z1toz3_z4_c1', 'z1toz3_z4_c2', 'z1toz3_z4_c3']].copy()

donuts_df = pd.DataFrame({
    'day_obs': np.array(aosTable['day_obs']),
    'seq_num': np.array(aosTable['seq_num']),
})
merged = donuts_df.merge(fit_keys, on=['day_obs', 'seq_num'], how='left')

k1 = merged['z1toz3_z4_c1'].to_numpy()
k2 = merged['z1toz3_z4_c2'].to_numpy()
k3 = merged['z1toz3_z4_c3'].to_numpy()
missing_fit = np.isnan(k1) | np.isnan(k2) | np.isnan(k3)
print(f'Donuts missing a linear fit: {int(missing_fit.sum())}')
keep &= ~missing_fit

Z4_linear_um = linear_fit_values(thx_deg, thy_deg, k1, k2, k3, fp_radius=FP_RADIUS_DEG)
# The fit was done on (Z4_data - Z4_pipeline_intrinsic), so the linear model
# should be subtracted from that residual. Equivalently, to get a Z4 that
# still has the intrinsic in it but with the low-order image-to-image drift
# removed, we subtract only Z4_linear:
Z4_linear_corr_um = Z4_data_um - Z4_linear_um
print('Z4 statistics (μm):')
for label, arr in [('Z4_data', Z4_data_um),
                   ('Z4_linear_model', Z4_linear_um),
                   ('Z4 - linear', Z4_linear_corr_um),
                   ('Z4_pipeline_intrinsic', Z4_pipeline_intrinsic_um)]:
    a = arr[keep]
    print(f'  {label:24s}  mean={np.nanmean(a):+.3f}  std={np.nanstd(a):.3f}  '
          f'n={len(a):,}')

In [ ]:
# Load the focal plane height map and build the KNN interpolator.
met_table = make_metrology_table(str(height_path))
print(f'Metrology points: {len(met_table):,}')
height_interp = get_height_interpolator(met_table, k=3, weight_type='distance',
                                        ztype='measured')

# Evaluate height at each donut (use intra-focal focal-plane position)
height_mm = height_interp(fpx, fpy)
print(f'Height range [mm]: [{np.nanmin(height_mm):+.4f}, {np.nanmax(height_mm):+.4f}]')

In [ ]:
# Custom Z4 intrinsic per donut:
#   Z4_custom = radial polynomial (i-band)  +  15 μm/mm * local piston height
r_deg = np.sqrt(thx_deg**2 + thy_deg**2)
Z4_radial_um = z4_radial_intrinsic_um(r_deg, Z4_POLY_PARAMS_IBAND)
Z4_height_um = HEIGHT_TO_Z4_UM_PER_MM * height_mm
Z4_custom_intrinsic_um = Z4_radial_um + Z4_height_um

print('Custom Z4 intrinsic components (μm):')
for label, arr in [('radial poly', Z4_radial_um),
                   ('height term', Z4_height_um),
                   ('custom total', Z4_custom_intrinsic_um)]:
    a = arr[keep]
    print(f'  {label:14s}  mean={np.nanmean(a):+.3f}  std={np.nanstd(a):.3f}  '
          f'min={np.nanmin(a):+.3f}  max={np.nanmax(a):+.3f}')

<a id='results'></a>
## Results & Plots

In [ ]:
# Common binning used across the focal-plane maps
xbins = np.linspace(-FP_MAX_MM, FP_MAX_MM, FP_NSTEPS)
ybins = np.linspace(-FP_MAX_MM, FP_MAX_MM, FP_NSTEPS)

xv = fpx[keep]
yv = fpy[keep]

In [ ]:
# 1) Median Z4 (linear-corrected) in focal plane — this is what the pipeline
#    intrinsic ought to match.
stat_corr, xe, ye = bin_median_2d(xv, yv, Z4_linear_corr_um[keep], xbins, ybins)
vabs = np.nanpercentile(np.abs(stat_corr), 98)
fig, ax = plot_fp_map(
    stat_corr, xe, ye,
    title='median Z4 − (k1,k2,k3) linear fit',
    cmap='RdBu_r', vmin=-vabs, vmax=vabs,
    label='Z4 [μm]',
)

In [ ]:
# 2) Custom Z4 intrinsic (radial polynomial + height term) in focal plane
stat_custom, _, _ = bin_median_2d(xv, yv, Z4_custom_intrinsic_um[keep], xbins, ybins)
vabs_c = np.nanpercentile(np.abs(stat_custom), 98)
fig, ax = plot_fp_map(
    stat_custom, xe, ye,
    title='custom Z4 intrinsic: radial poly + height map',
    cmap='RdBu_r', vmin=-vabs_c, vmax=vabs_c,
    label='Z4 [μm]',
)

In [ ]:
# 3) Difference: median (Z4 − linear) − median (custom Z4 intrinsic)
#    If the pipeline Z4 intrinsic already absorbs the CCD height variation,
#    this should look flat; any CCD-to-CCD pattern highlights what's left.
diff = stat_corr - stat_custom
vabs_d = np.nanpercentile(np.abs(diff), 98)
fig, ax = plot_fp_map(
    diff, xe, ye,
    title='median (Z4 − linear)  −  custom Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_d, vmax=vabs_d,
    label='ΔZ4 [μm]',
)

### Pipeline Z4 intrinsic vs custom Z4 intrinsic

Directly compare the per-donut Z4 intrinsic already stored in the HDF5 (computed
during fitting) against the custom Z4 intrinsic built from the radial polynomial
plus the CCD height term.

In [ ]:
# 4a) Pipeline Z4 intrinsic map (μm) for reference
stat_pipe, _, _ = bin_median_2d(xv, yv, Z4_pipeline_intrinsic_um[keep], xbins, ybins)
vabs_p = np.nanpercentile(np.abs(stat_pipe), 98)
fig, ax = plot_fp_map(
    stat_pipe, xe, ye,
    title='pipeline Z4 intrinsic (from HDF5 zk_intrinsic)',
    cmap='RdBu_r', vmin=-vabs_p, vmax=vabs_p,
    label='Z4 [μm]',
)

# 4b) Pipeline − custom, in focal plane
diff_pipe = stat_pipe - stat_custom
vabs_pd = np.nanpercentile(np.abs(diff_pipe), 98)
fig, ax = plot_fp_map(
    diff_pipe, xe, ye,
    title='pipeline Z4 intrinsic  −  custom Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_pd, vmax=vabs_pd,
    label='ΔZ4 [μm]',
)

In [ ]:
# 4c) Hexbin: pipeline vs custom (1:1 line overlaid)
pipe = Z4_pipeline_intrinsic_um[keep]
custom = Z4_custom_intrinsic_um[keep]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
hb = ax.hexbin(pipe, custom, gridsize=120, mincnt=1, cmap='viridis')
lo = min(np.nanmin(pipe), np.nanmin(custom))
hi = max(np.nanmax(pipe), np.nanmax(custom))
ax.plot([lo, hi], [lo, hi], 'r-', lw=0.8, label='y = x')
ax.set_xlabel('pipeline Z4 intrinsic [μm]')
ax.set_ylabel('custom Z4 intrinsic [μm]')
ax.set_title('pipeline vs custom Z4 intrinsic (per donut)')
ax.legend(loc='upper left', frameon=True)
add_colorbar(hb).set_label('count')
fig.tight_layout()

# 4d) Hexbin: difference vs height (diagnostic for the height term)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
hb = ax.hexbin(height_mm[keep], (pipe - custom), gridsize=120, mincnt=1,
               cmap='viridis')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('local height [mm]')
ax.set_ylabel('pipeline − custom Z4 intrinsic [μm]')
ax.set_title('Does the pipeline intrinsic track the height map?')
add_colorbar(hb).set_label('count')
fig.tight_layout()

# 4e) Residuals of the linear-corrected data against each intrinsic
res_pipe = Z4_linear_corr_um[keep] - pipe
res_cust = Z4_linear_corr_um[keep] - custom
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, data, name in zip(axes, [res_pipe, res_cust],
                          ['(Z4 − linear) − pipeline intrinsic',
                           '(Z4 − linear) − custom intrinsic']):
    ax.hist(data, bins=200, histtype='step')
    ax.set_title(f'{name}\nmean={np.nanmean(data):+.3f} μm,  '
                 f'std={np.nanstd(data):.3f} μm')
    ax.set_xlabel('ΔZ4 [μm]')
axes[0].set_ylabel('donuts')
fig.tight_layout()